# FinNutriAgent — Reproducible Computational Experiment

**FinNutriAgent: Agentic AI for Household Nutrition and Budget Optimization**

| | |
|---|---|
| Author | Ali Akarma |
| ORCID | https://orcid.org/0009-0002-6687-9380 |
| Repository | https://github.com/aliakarma/FinNutriAgent |
| DOI | https://doi.org/10.5281/zenodo.18849993 |

---

This notebook demonstrates the **FinNutriAgent framework**, an agentic
decision-support system that optimizes household food purchasing under
financial and nutritional constraints.

Pipeline:

1. Budget Agent — computes weekly household food budgets
2. Nutrition Agent — aggregates household nutritional requirements
3. Price Agent — evaluates market food prices
4. MILP Optimization — generates a cost-efficient weekly meal plan

The optimization ensures:

- nutritional adequacy
- budget constraints
- halal food filtering
- food diversity constraints

All results are fully reproducible using datasets included in the repository.


In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 1 — Environment Setup
# ─────────────────────────────────────────────────────────────

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec

from pulp import (
    LpProblem,
    LpMinimize,
    LpVariable,
    lpSum,
    LpStatus,
    PULP_CBC_CMD
)

np.random.seed(42)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)

plt.rcParams.update({"figure.dpi":120})

print("Environment ready.")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 2 — Experiment Metadata
# ─────────────────────────────────────────────────────────────

from datetime import datetime

EXPERIMENT = {
    "project": "FinNutriAgent",
    "author": "Ali Akarma",
    "orcid": "0009-0002-6687-9380",
    "timestamp": datetime.utcnow().isoformat(),
    "python": sys.version
}

for k,v in EXPERIMENT.items():
    print(f"{k}: {v}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Section 3 — Data Loading
# ─────────────────────────────────────────────────────────────

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

FINANCIAL_PATH = DATA_DIR / "financial" / "financial_data.csv"
FOOD_PATH = DATA_DIR / "composition" / "food_composition.csv"
PRICES_PATH = DATA_DIR / "prices" / "food_prices.csv"
NUTRITION_PATH = DATA_DIR / "nutrition" / "nutrition_requirements.csv"

for f in [FINANCIAL_PATH,FOOD_PATH,PRICES_PATH,NUTRITION_PATH]:
    if not f.exists():
        raise FileNotFoundError(f"Missing dataset: {f}")

financial_df = pd.read_csv(FINANCIAL_PATH)
food_df = pd.read_csv(FOOD_PATH)
prices_df = pd.read_csv(PRICES_PATH)
nutrition_df = pd.read_csv(NUTRITION_PATH)

print("Datasets loaded successfully.")

print(len(financial_df),"financial rows")
print(len(food_df),"foods")
print(len(prices_df),"price entries")
print(len(nutrition_df),"nutrition records")

## Budget Agent

In [ ]:
EXPENSE_COLS = ['rent','utilities','transport','education','healthcare','savings_target']

def get_weekly_budget(user_id):
    row = financial_df[financial_df.user_id==user_id].iloc[0]
    residual = row['monthly_income'] - sum(row[c] for c in EXPENSE_COLS)
    return residual/4.33

## Nutrition Agent

In [ ]:
def get_weekly_targets(person_ids):

    subset = nutrition_df[nutrition_df.person_id.isin(person_ids)]

    return {
        "calories": subset.calories_kcal.sum()*7,
        "protein": subset.protein_g.sum()*7,
        "vitamin_d": subset.vitamin_d_mcg.sum()*7,
        "iron": subset.iron_mg.sum()*7
    }

## Price Agent

In [ ]:
avg_prices = prices_df.groupby("food_id")["price_per_kg"].mean().reset_index()

merged_df = food_df.merge(avg_prices,on="food_id")

merged_df["cost_per_gram"] = merged_df.price_per_kg/1000

## Optimization Engine

In [ ]:
NUTRIENT_MAP={
 "calories":"calories_per_100g",
 "protein":"protein_g",
 "vitamin_d":"vitamin_d_mcg",
 "iron":"iron_mg"
}

In [ ]:
def run_optimization(weekly_budget,nutrient_targets,food_df):

    ids=food_df.food_id.tolist()

    cost={r.food_id:r.price_per_kg/1000 for _,r in food_df.iterrows()}

    x={f:LpVariable(f"x_{f}",lowBound=0) for f in ids}

    prob=LpProblem("FinNutriAgent",LpMinimize)

    prob+=lpSum(x[f]*cost[f] for f in ids)

    for nut,col in NUTRIENT_MAP.items():

        nv=food_df.set_index("food_id")[col].to_dict()

        prob+=lpSum(x[f]*nv[f]/100 for f in ids)>=nutrient_targets[nut]

    prob+=lpSum(x[f]*cost[f] for f in ids)<=weekly_budget

    prob.solve(PULP_CBC_CMD(msg=0))

    status=LpStatus[prob.status]

    if status!="Optimal":
        raise RuntimeError("Optimization failed")

    rows=[]

    for f in ids:
        v=x[f].varValue or 0
        if v>0:
            rows.append({
                "food":f,
                "grams":v
            })

    return pd.DataFrame(rows)

## Run Example Household

In [ ]:
USER="U017"

budget=get_weekly_budget(USER)

targets=get_weekly_targets(["P1","P2","P3","P4"])

start=time.time()

result=run_optimization(budget,targets,merged_df)

runtime=time.time()-start

print("Optimization runtime:",runtime)

result.head()

## Export Results

In [ ]:
results_dir=PROJECT_ROOT/"results"

results_dir.mkdir(exist_ok=True)

result.to_csv(results_dir/"meal_plan.csv",index=False)

print("Results saved.")